# w9_scale_query.ipynb — 容量/读出网格 slot{N}i2ce{mean|line}

User: 直接在 i2ce 下测 {q_num 4,8,16} x {mean-pool, linear}. Two orthogonal
questions in one 2x3 grid: does MORE query slots help (capacity), and does a
LEARNED linear pool over the slots beat the plain mean (read-out)? Loss is
plain i2ce everywhere -- only SetPoolN's slot count and the slot->DM pooling
change; the linear pool is INITIALIZED to equal mean (block [I/N ...]) so any
gain is pure signal. (4,mean) == the existing i2ce tower (reference row, NOT
rerun); the 5 new towers are (4,line),(8,mean),(8,line),(16,mean),(16,line).
@512/2000ep, ZS-only, rvsel selection. Each ~5.6G/~3h; all 5 pack on one
A100 (VRAM scheduler measures per arm). Refs: i2ce@512 (=slot4mean) &
ce@512. AUTO-STOPS.


In [ ]:
# constants
import os

REPO = os.path.abspath("..")   # this release folder (contains Pod/ and VICReg_review/)
DATA_SRC = "/workspace/fusion_cache_w9"
DATA_RAM = "/dev/shm/fusion_cache_w9"
OUT_DIR = "/workspace/w9_out"          # fixed-split campaign dir

SAFETY = 0.85
RESERVE_GIB = 1.5

# (arm, cap, epochs). 5 new cells; (4,mean)=i2ce is the reference, not rerun.
GRID = [(4, "line"), (8, "mean"), (8, "line"), (16, "mean"), (16, "line")]
FLASH = [(f"wcle_slot{n}i2ce{pl}_icetf", 512, 2000) for n, pl in GRID]
os.makedirs(OUT_DIR, exist_ok=True)
print(f"{len(FLASH)} new towers:", [f"slot{n}i2ce{pl}" for n, pl in GRID])


In [ ]:
# Local setup (release build: the code ships with this folder -- no
# repository synchronisation is needed or performed).
import importlib.util
import os
import sys
for pkg in ("sklearn", "scipy"):
    if importlib.util.find_spec(pkg) is None:
        %pip -q install scikit-learn scipy
        break
os.chdir(REPO)
sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "Pod"))
import w9_jobs as J
print("machinery loaded")

In [ ]:
# Stage the corpus into RAM (llm views not needed).
import shutil
from pathlib import Path
REQUIRED = ["games.npz", "wiki_eval.npz", "wscan_gal_rev.npz",
            "wscan_pool_rev.npy", "wscan_pool_rev_rid.npy", "wscan_pool_rev_len.npy",
            "ss_queries_rev.npz", "ss_queries_rev_S.npy",
            "wiki_clean_views.npz", "sp_raw_views.npz",
            "tag_labels.npz",
            "wiki_eval_split.json", "_tag_splitM.json"]
src = Path(DATA_SRC)
missing = [f for f in REQUIRED if not (src / f).exists()]
assert not missing, f"missing in {DATA_SRC}: {missing}"
dst = Path(DATA_RAM)
dst.mkdir(parents=True, exist_ok=True)
for f in REQUIRED:
    s, d = src / f, dst / f
    if not d.exists() or d.stat().st_size != s.stat().st_size:
        print(f"staging {f} ({s.stat().st_size/1e9:.2f} GB) ...", flush=True)
        shutil.copyfile(s, d)
DATA_DIR = str(dst)
print("corpus in RAM:", DATA_DIR)


In [ ]:
# Full pool: must be READY on the volume; stage onto fast local storage.
import os, time
from pathlib import Path
from Pod.h5_staging import parallel_copy

ready = Path(DATA_SRC) / "full_pool_READY"
assert ready.exists(), "full pool not READY -- run a campaign notebook's build cell once"
src_v = Path(DATA_SRC) / "full_pool_fp16.npy"
src_m = Path(DATA_SRC) / "full_pool_meta.npz"
need = src_v.stat().st_size + (5 << 30)

def _free(p):
    st = os.statvfs(p)
    return st.f_bavail * st.f_frsize

dest_dir = None
for cand in ("/dev/shm", "/root/data", "/root"):
    Path(cand).mkdir(parents=True, exist_ok=True)
    if _free(cand) > need:
        dest_dir = Path(cand)
        break
if dest_dir is None:
    print("WARNING: no local space -- workers will mmap the NETWORK VOLUME copy.")
    FULL_POOL_PATH = str(src_v)
else:
    dst_v = dest_dir / "full_pool_fp16.npy"
    if dst_v.exists() and dst_v.stat().st_size == src_v.stat().st_size:
        print("local full pool already staged:", dst_v)
    else:
        t0 = time.time()
        tmp = dst_v.with_name(dst_v.name + ".copying")
        print(f"staging {src_v.stat().st_size/2**30:.0f} GiB -> {dst_v} ...", flush=True)
        parallel_copy(src_v, tmp, workers=8)
        os.replace(tmp, dst_v)
        print(f"staged in {(time.time()-t0)/60:.1f} min", flush=True)
    import shutil
    shutil.copyfile(src_m, dest_dir / "full_pool_meta.npz")
    FULL_POOL_PATH = str(dst_v)
print("FULL_POOL_PATH =", FULL_POOL_PATH)


In [ ]:
# VRAM-BUDGET DYNAMIC SCHEDULER (user: warmup once, measure each cap's real
# peak, keep a now_used pool per GPU, pack heterogeneously). Warmup runs the
# REAL worker step at each cap so the backward-pass 4x-view loss matrices are
# in the measured peak. Jobs are placed on the least-loaded GPU that fits;
# many 512s stack on one A100, 512+1024 co-reside, 4096 runs solo. A job whose
# cost exceeds every budget still runs SOLO on an empty GPU (no deadlock).
import os, subprocess, tempfile, threading, time
from pathlib import Path

def _norm(e):
    return (e[0], e[1], e[2] if len(e) == 3 else SCALE_EPOCHS)

cdir = Path(OUT_DIR) / "claims"
logd = Path(OUT_DIR) / "logs"
logd.mkdir(parents=True, exist_ok=True)
# measure runs write to a LOCAL scratch dir: with the real job name on the
# SHARED volume, a measure could load another machine's resume bundle
# (start_ep >= 1 -> zero steps -> no cost file) or race its zs_traj writes.
MEAS_OUT = os.path.join(tempfile.gettempdir(), "w9_measure_out")
os.makedirs(MEAS_OUT, exist_ok=True)
gpus = J.detect_gpus()

def _smi_mib(field, g):
    out = subprocess.check_output(
        ["nvidia-smi", f"--query-gpu={field}", "--format=csv,noheader,nounits",
         "-i", str(g)]).decode().strip().split("\n")[0]
    return int(out) * 2**20

free = {g: _smi_mib("memory.free", g) for g in gpus}
budget = {g: int(free[g] * SAFETY - RESERVE_GIB * 2**30) for g in gpus}
vram_gib = min(free.values()) / 2**30
CAP_CEIL = globals().get("MAX_CAP_48G", 10**9) if vram_gib < 60 else 10**9
print(f"[vram] free/GPU ~{vram_gib:.0f}G  budgets "
      f"{[f'{budget[g] / 2**30:.0f}G' for g in gpus]}  cap-ceil "
      f"{CAP_CEIL if CAP_CEIL < 10**9 else 'none'}")

# --- warmup: peak bytes per distinct cap (heaviest arm at that cap) ---
grid = [_norm(e) for e in FLASH]
caps = sorted({cap for _a, cap, _e in grid if cap <= CAP_CEIL})
cost = {}
for cap in caps:
    arms_c = [a for a, c, _e in grid if c == cap]
    arm = next((a for a in arms_c if "exp" in a), arms_c[0])   # heaviest proj
    tf = Path(tempfile.gettempdir()) / f"w9vram_{cap}.txt"
    tf.unlink(missing_ok=True)
    cmd = ["python", "-u", J.FS_WORKER, "--data-dir", DATA_DIR, "--out-dir",
           MEAS_OUT, "--repo", REPO, "--arm", arm, "--anchor-cap", str(cap),
           "--epochs", "1", "--full-pool", "--full-pool-path", FULL_POOL_PATH,
           "--measure-vram", str(tf)]
    print(f"[warmup] cap {cap} via {arm} ...", flush=True)
    with open(logd / f"measure_g{cap}.log", "w") as fh:
        subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT,
                       env=dict(os.environ, CUDA_VISIBLE_DEVICES=gpus[0]))
    cost[cap] = int(tf.read_text()) if tf.exists() else budget[gpus[0]] + 1
    print(f"[warmup] cap {cap}: {cost[cap] / 2**30:.2f}G peak", flush=True)

# --- job list (skip done + cap ceiling), best-fit decreasing ---
todo = []
for arm, cap, ep in grid:
    nm = J.fs_label(arm, cap, False, 0, "clean", 16)
    if cap > CAP_CEIL:
        print(f"[skip-vram] {nm} cap {cap} > {CAP_CEIL}"); continue
    if (Path(OUT_DIR) / f"tower_{nm}_fp_ep{ep}.npz").exists():
        print(f"[skip] {nm} at {ep}"); continue
    todo.append((arm, cap, ep, nm, cost[cap]))
todo.sort(key=lambda j: -j[4])

# --- dynamic pool scheduler ---
now_used = {g: 0 for g in gpus}
fails = []
cv = threading.Condition()

def run_job(g, arm, cap, ep, nm, c):
    try:
        if not J.try_claim(cdir, nm):
            print(f"[claim] {nm} held elsewhere -- skipped", flush=True); return
        cmd = ["python", "-u", J.FS_WORKER, "--data-dir", DATA_DIR, "--out-dir",
               OUT_DIR, "--repo", REPO, "--arm", arm, "--anchor-cap", str(cap),
               "--epochs", str(ep), "--ckpt-every", str(J.CKPT_EVERY),
               "--ckpt-seeds", str(J.FS_CKPT_SEEDS),
               "--topup-seeds", str(J.TOPUP_SEEDS),
               "--full-pool", "--full-pool-path", FULL_POOL_PATH,
               "--claim-file", str(cdir / f"{nm}.claim")]
        t0 = time.time()
        with open(logd / f"{arm}_g{cap}.log", "w") as fh:
            p = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT,
                               env=dict(os.environ, CUDA_VISIBLE_DEVICES=g))
        if p.returncode != 0:
            (cdir / f"{nm}.claim").unlink(missing_ok=True); fails.append(nm)
        print(f"[gpu{g}] {'ok' if p.returncode == 0 else 'FAIL'} {nm} @{ep} "
              f"[{(time.time() - t0) / 60:.1f}m]", flush=True)
    finally:
        with cv:
            now_used[g] -= c
            cv.notify_all()

stop_evt = threading.Event()
threading.Thread(target=J._monitor, args=([logd], stop_evt), daemon=True).start()
active = []
with cv:
    pending = list(todo)
    while pending or any(t.is_alive() for t in active):
        progressed = False
        i = 0
        while i < len(pending):
            arm, cap, ep, nm, c = pending[i]
            fit = [g for g in gpus if now_used[g] + c <= budget[g] or now_used[g] == 0]
            if not fit:
                i += 1; continue
            g = min(fit, key=lambda g: now_used[g])   # least-loaded -> spread
            now_used[g] += c
            th = threading.Thread(target=run_job,
                                  args=(g, arm, cap, ep, nm, c), daemon=True)
            active.append(th); th.start(); pending.pop(i)
            print(f"[sched] {nm} -> gpu{g}  ({c / 2**30:.1f}G, used "
                  f"{now_used[g] / 2**30:.1f}/{budget[g] / 2**30:.0f}G)", flush=True)
            progressed = True
        active = [t for t in active if t.is_alive()]
        if not progressed:
            cv.wait(timeout=3)
stop_evt.set()
for t in active:
    t.join()
print(f"drained; {len(fails)} failed")
for nm in fails:
    print("  FAILED:", nm)


In [ ]:
# Readout: the 2x3 capacity/read-out grid, ZSbest-primary (rvsel).
import json
import numpy as np
from pathlib import Path
VORD = ["neutral", "noname", "positive", "negative"]
ROWS = [("wcle_i2ce_icetf", "slot4 mean (=i2ce ref)"),
        ("wcle_slot4i2celine_icetf", "slot4 line"),
        ("wcle_slot8i2cemean_icetf", "slot8 mean"),
        ("wcle_slot8i2celine_icetf", "slot8 line"),
        ("wcle_slot16i2cemean_icetf", "slot16 mean"),
        ("wcle_slot16i2celine_icetf", "slot16 line"),
        ("wcle_ce_cetf", "ce (no-I ref)")]
def _row(lab, nm):
    zb = Path(OUT_DIR) / f"zsbest_{nm}_fp.json"
    zp = Path(OUT_DIR) / f"zs_traj_{nm}_fp.json"
    ft = Path(OUT_DIR) / f"ft4var_{nm}_fp_best.json"
    if zb.exists():
        d = json.loads(zb.read_text())
        m4z = np.mean([d["nm_" + v] for v in VORD])
        line = (f"{lab:30s} ZSbest@ep{d['best_ep']:>4}(val) "
                + " ".join(f"{v[:3]}:{d['nm_' + v]:.3f}" for v in VORD)
                + f" m4z:{m4z:.3f} tag:{d['tag_neutral']:.3f}/{d['tag_noname']:.3f}")
    elif zp.exists():
        tr = json.loads(zp.read_text())
        eps = sorted(tr, key=lambda k: int(k[2:]))
        pk = max(eps, key=lambda k: tr[k]["nm_neutral"])
        line = (f"{lab:30s} ZS test-peak*@{pk[2:]:>4} neu {tr[pk]['nm_neutral']:.3f}"
                f" non {tr[pk]['nm_noname']:.3f}"
                f" tag {tr[pk]['tag_neutral']:.3f}/{tr[pk]['tag_noname']:.3f}")
    else:
        return f"{lab:30s} (pending)"
    if ft.exists():
        d2 = json.loads(ft.read_text())
        m4 = np.mean([np.mean([x[v]["h1"] for x in d2["per_seed"]]) for v in VORD])
        line += f" | FT m4 {m4:.3f}"
    return line

for arm, lab in ROWS:
    print(_row(lab, f"w9_{arm}"))


In [ ]:
# AUTO-STOP removed in the release build: stopping the machine is cloud-
# provider tooling, not part of the experiment. All results are already on
# the shared volume when the run cells finish.
print("run complete -- results are in", OUT_DIR)